<a href="https://colab.research.google.com/github/aMDy0k/workspace/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###### import

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sympy import *
import requests
import json
import time
import os
import ast

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Загружаем датафрейм
try:
  df = pd.read_json("/content/drive/MyDrive/core/df/dota2.json")

except:
  FileNotFoundError

In [12]:
df

,match_seq_num,radiant_win,start_time,duration,lobby_type,game_mode,avg_rank_tier,num_rank_tier,cluster,radiant_team,dire_team
0,7558648158,False,1789298862,943,0,22,35,4,145,"[20, 91, 90, 145, 96]","[8, 18, 52, 55, 10]"
1,7558648166,True,1789298792,980,7,22,54,1,187,"[73, 37, 11, 112, 36]","[32, 96, 10, 18, 44]"
2,7558645563,False,1789298746,912,0,22,63,5,144,"[18, 37, 96, 53, 26]","[2, 90, 81, 138, 6]"
3,7558648566,False,1789298737,1036,0,22,33,3,184,"[39, 121, 10, 96, 35]","[101, 120, 113, 20, 128]"
4,7558644977,False,1789298690,928,7,22,41,3,423,"[88, 123, 72, 155, 110]","[77, 11, 74, 73, 46]"
5,7558644599,False,1789298691,935,7,22,33,2,182,"[93, 29, 58, 98, 84]","[36, 145, 91, 74, 59]"
6,7558644746,True,1789298686,943,7,22,22,3,271,"[14, 11, 45, 37, 38]","[136, 20, 119, 96, 73]"
7,7558648371,False,1789298661,1084,7,22,61,5,181,"[86, 61, 75, 54, 28]","[57, 93, 27, 145, 77]"
8,7558644958,True,1789298645,933,7,22,65,7,141,"[42, 93, 85, 26, 25]","[126, 11, 22, 76, 54]"
9,7558643255,True,1789298591,937,7,22,54,9,272,"[76, 54, 86, 30, 5]","[29, 80, 33, 35, 110]"


###### словарь

In [3]:
heroes = {
    1: "Anti-Mage", 2: "Axe", 3: "Bane", 4: "Bloodseeker", 5: "Crystal Maiden",
    6: "Drow Ranger", 7: "Earthshaker", 8: "Juggernaut", 9: "Mirana", 10: "Morphling",
    11: "Shadow Fiend", 12: "Phantom Lancer", 13: "Puck", 14: "Pudge", 15: "Razor",
    16: "Sand King", 17: "Storm Spirit", 18: "Sven", 19: "Tiny", 20: "Vengeful Spirit",
    21: "Windranger", 22: "Zeus", 23: "Kunkka", 25: "Lina", 26: "Lion",
    27: "Shadow Shaman", 28: "Slardar", 29: "Tidehunter", 30: "Witch Doctor", 31: "Lich",
    32: "Riki", 33: "Enigma", 34: "Tinker", 35: "Sniper", 36: "Necrophos",
    37: "Warlock", 38: "Beastmaster", 39: "Queen of Pain", 40: "Venomancer", 41: "Faceless Void",
    42: "Wraith King", 43: "Death Prophet", 44: "Phantom Assassin", 45: "Pugna", 46: "Templar Assassin",
    47: "Viper", 48: "Luna", 49: "Dragon Knight", 50: "Dazzle", 51: "Clockwerk",
    52: "Leshrac", 53: "Nature's Prophet", 54: "Lifestealer", 55: "Dark Seer", 56: "Clinkz",
    57: "Omniknight", 58: "Enchantress", 59: "Huskar", 60: "Night Stalker", 61: "Broodmother",
    62: "Bounty Hunter", 63: "Weaver", 64: "Jakiro", 65: "Batrider", 66: "Chen",
    67: "Spectre", 68: "Ancient Apparition", 69: "Doom", 70: "Ursa", 71: "Spirit Breaker",
    72: "Gyrocopter", 73: "Alchemist", 74: "Invoker", 75: "Silencer", 76: "Outworld Destroyer",
    77: "Lycan", 78: "Brewmaster", 79: "Shadow Demon", 80: "Lone Druid", 81: "Chaos Knight",
    82: "Meepo", 83: "Treant Protector", 84: "Ogre Magi", 85: "Undying", 86: "Rubick",
    87: "Disruptor", 88: "Nyx Assassin", 89: "Naga Siren", 90: "Keeper of the Light", 91: "Io",
    92: "Visage", 93: "Slark", 94: "Medusa", 95: "Troll Warlord", 96: "Centaur Warrunner",
    97: "Magnus", 98: "Timbersaw", 99: "Bristleback", 100: "Tusk", 101: "Skywrath Mage",
    102: "Abaddon", 103: "Elder Titan", 104: "Legion Commander", 105: "Techies", 106: "Ember Spirit",
    107: "Earth Spirit", 108: "Underlord", 109: "Terrorblade", 110: "Phoenix", 111: "Oracle",
    112: "Winter Wyvern", 113: "Arc Warden", 114: "Monkey King", 119: "Dark Willow", 120: "Pangolier",
    121: "Grimstroke", 123: "Hoodwink", 126: "Void Spirit", 128: "Snapfire", 129: "Mars",
    131: "Muerta", 135: "Dawnbreaker", 136: "Marci", 137: "Primal Beast",
    138: "Ringmaster", 145: "Kez", 155: "Largo"
}

### ___

In [4]:
# Если df пустой или еще не создан — стартуем с текущей минуты (None)
if "df" in locals() and not df.empty and "match_id" in df.columns:
    min_id = df["match_id"].min()

    # Проверяем, что минимум — это реальное число, а не NaN или 0
    if pd.notna(min_id) and min_id > 1000000000:
        less_than_match_id = int(min_id)
        print(f"Парсер продолжит сбор, начиная с match_id < {less_than_match_id}")
    else:
        less_than_match_id = None
        print("Парсер начинает сбор с самых актуальных матчей.")
else:
    less_than_match_id = None
    print("Парсер начинает сбор с самых актуальных матчей.")


Парсер начинает сбор с самых актуальных матчей.


In [5]:
# Базовый URL для получения СЫРЫХ публичных матчей
url = "https://api.opendota.com/api/publicMatches"

# Сюда мы будем собирать наш огромный сырой датасет
all_raw_matches = []

print("Начинаем выкачивать сырой датасет напрямую...")

for i in range(50):
    params = {}
    if less_than_match_id:
        params['less_than_match_id'] = less_than_match_id

    try:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            batch = response.json()
            if not batch:
                print("Матчи закончились.")
                break

            all_raw_matches.extend(batch)
            print(f"Скачана пачка {i+1}. Всего матчей в датасете: {len(all_raw_matches)}")

            # Запоминаем ID самого старого матча в пачке, чтобы следующая пачка пошла дальше в прошлое
            less_than_match_id = batch[-1]['match_id']

            # Защита от бана: спим 1 секунду между запросами (лимит OpenDota - 60 запросов в минуту)
            time.sleep(1)
        else:
            print(f"Ошибка. Статус: {response.status_code}, Ответ: {response.text}")
            break
    except Exception as e:
        print(f"Ошибка сети: {e}")
        break

# Сохраняем собранный сырой массив в файл
if all_raw_matches:
    filename = "raw_matches_dataset.json"
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(all_raw_matches, f, indent=4, ensure_ascii=False)

    print(f"\n  Сырой датасет сохранен в файл: '{filename}'")
    print("Структура одного сырого матча:")
    print(json.dumps(all_raw_matches[0], indent=4))
else:
    print("Не удалось собрать данные.")


Начинаем выкачивать сырой датасет напрямую...
Скачана пачка 1. Всего матчей в датасете: 100
Скачана пачка 2. Всего матчей в датасете: 200
Скачана пачка 3. Всего матчей в датасете: 300
Скачана пачка 4. Всего матчей в датасете: 400
Скачана пачка 5. Всего матчей в датасете: 500
Скачана пачка 6. Всего матчей в датасете: 600
Скачана пачка 7. Всего матчей в датасете: 700
Скачана пачка 8. Всего матчей в датасете: 800
Скачана пачка 9. Всего матчей в датасете: 900
Скачана пачка 10. Всего матчей в датасете: 1000
Ошибка. Статус: 429, Ответ: {"error":"minute rate limit exceeded"}

  Сырой датасет сохранен в файл: 'raw_matches_dataset.json'
Структура одного сырого матча:
{
    "match_id": 8996927401,
    "match_seq_num": 7558647421,
    "radiant_win": false,
    "start_time": 1789299917,
    "duration": 0,
    "lobby_type": 14,
    "game_mode": 13,
    "avg_rank_tier": 62,
    "num_rank_tier": 1,
    "cluster": 153,
    "radiant_team": [
        0,
        0,
        0,
        0,
        0
    ],


In [6]:
def clean_teams_in_row(row):
    def parse_value(val):
        if isinstance(val, str):
            try:
                return json.loads(val)
            except:
                try:
                    return ast.literal_eval(val)
                except:
                    return [
                        int(x) for x in val.split(",") if x.strip().isdigit()
                    ]
        return [int(x) for x in val] if isinstance(val, list) else []

    row["radiant_team"] = parse_value(row.get("radiant_team", []))
    row["dire_team"] = parse_value(row.get("dire_team", []))
    return row

In [7]:
if all_raw_matches:
    new_df = pd.DataFrame(all_raw_matches)

    if "match_id" in new_df.columns:
        new_df = new_df.set_index("match_id")

    df = pd.concat([df, new_df], axis=0)\
      if ("df" in locals() and not df.empty) else new_df

    df = df[~df.index.duplicated(keep="first")]

    df = df.apply(clean_teams_in_row, axis=1)

    df = df[(df["game_mode"] == 22) & (df["duration"] > (15 * 60))]

    print(f"📊 Датасет успешно увеличен! Текущий размер: {df.shape[0]} уникальных матчей.\n")
else:
    print("Новых матчей не собрано.")

📊 Датасет успешно увеличен! Текущий размер: 42 уникальных матчей.



In [8]:
df.to_json(
    "/content/drive/MyDrive/core/df/dota2.json", orient="records", indent=4
)

Файл успешно сохранен на Google Диск!


In [9]:
import pandas as pd

# Быстрая проверка структуры
print("=== ЭКСПРЕСС-ПРОВЕРКА ДАТАСЕТА ===")
print(f" Всего матчей в таблице: {df.shape[0]}")
print(f" Количество столбцов: {df.shape[1]}")

# 1. Проверяем индекс
is_match_id_index = df.index.name == 'match_id' or (df.index.astype(str).str.len() > 8).all()
print(f" Индекс является 'match_id': {' SUCCESS' if is_match_id_index else '❌ FAILED (Индекс сбился)'}")
print(f" Пример текущего индекса (первых 3 строки): {list(df.index[:3])}")

# 2. Проверяем отсутствие дубликатов
dup_count = df.index.duplicated().sum()
print(f" Найденные дубликаты матчей: {dup_count} ({' SUCCESS' if dup_count == 0 else '❌ FAILED'})")

# 3. Проверяем типы данных в командах
first_rad = df['radiant_team'].iloc[0] if not df.empty else None
is_clean_list = isinstance(first_rad, list) and len(first_rad) > 0 and isinstance(first_rad[0], int)
print(f" Формат героев в ячейках: {' SUCCESS (Чистые списки чисел)' if is_clean_list else '❌ FAILED (Всё еще текст)'}")

# 4. Проверяем среднее количество героев через правильный метод .apply(len)
if is_clean_list:
    rad_len = df['radiant_team'].apply(len).mean()
    dire_len = df['dire_team'].apply(len).mean()
    print(f" Среднее количество героев в матче: Radiant={rad_len:.2f}, Dire={dire_len:.2f} ({' SUCCESS' if rad_len == 5.0 and dire_len == 5.0 else '⚠️ Внимание, есть неполные составы!'})")

# 5. Проверяем фильтры режима и времени
min_duration = df['duration'].min() / 60
unique_modes = df['game_mode'].unique()
print(f" Минимальная длительность матча: {min_duration:.1f} мин ({' SUCCESS' if min_duration >= 15 else '❌ Есть слишком короткие матчи'})")
print(f" Уникальные режимы игры в базе: {unique_modes} ({' SUCCESS' if list(unique_modes) == [22] else '❌ Есть другие режимы кроме All Pick (22)'})")
print("==================================")


=== ЭКСПРЕСС-ПРОВЕРКА ДАТАСЕТА ===
 Всего матчей в таблице: 42
 Количество столбцов: 11
 Индекс является 'match_id':  SUCCESS
 Пример текущего индекса (первых 3 строки): [8996897532, 8996895665, 8996894644]
 Найденные дубликаты матчей: 0 ( SUCCESS)
 Формат героев в ячейках:  SUCCESS (Чистые списки чисел)
 Среднее количество героев в матче: Radiant=5.00, Dire=5.00 ( SUCCESS)
 Минимальная длительность матча: 15.2 мин ( SUCCESS)
 Уникальные режимы игры в базе: [22] ( SUCCESS)


In [10]:
df

,match_seq_num,radiant_win,start_time,duration,lobby_type,game_mode,avg_rank_tier,num_rank_tier,cluster,radiant_team,dire_team
match_id,,,,,,,,,,,
8996897532,7558648158,False,1789298862,943,0,22,35,4,145,"[20, 91, 90, 145, 96]","[8, 18, 52, 55, 10]"
8996895665,7558648166,True,1789298792,980,7,22,54,1,187,"[73, 37, 11, 112, 36]","[32, 96, 10, 18, 44]"
8996894644,7558645563,False,1789298746,912,0,22,63,5,144,"[18, 37, 96, 53, 26]","[2, 90, 81, 138, 6]"
8996894190,7558648566,False,1789298737,1036,0,22,33,3,184,"[39, 121, 10, 96, 35]","[101, 120, 113, 20, 128]"
8996892945,7558644977,False,1789298690,928,7,22,41,3,423,"[88, 123, 72, 155, 110]","[77, 11, 74, 73, 46]"
8996892818,7558644599,False,1789298691,935,7,22,33,2,182,"[93, 29, 58, 98, 84]","[36, 145, 91, 74, 59]"
8996892783,7558644746,True,1789298686,943,7,22,22,3,271,"[14, 11, 45, 37, 38]","[136, 20, 119, 96, 73]"
8996891987,7558648371,False,1789298661,1084,7,22,61,5,181,"[86, 61, 75, 54, 28]","[57, 93, 27, 145, 77]"
8996891927,7558644958,True,1789298645,933,7,22,65,7,141,"[42, 93, 85, 26, 25]","[126, 11, 22, 76, 54]"
